# 20장 다중 에이전트와 AutoGen

이 장은 PDF 교재의 LLM Agent, LangChain, LangGraph 흐름을 바탕으로 여러 LLM 에이전트가 역할을 나누어 협업하는 구조를 정리합니다.

앞 장들에서는 하나의 LLM 체인이나 하나의 챗봇이 질문에 답하는 구조를 다뤘습니다. 다중 에이전트 구조에서는 여러 에이전트가 각자 역할을 맡고, 서로 대화하거나 결과를 검토하며 문제를 해결합니다. AutoGen은 이런 다중 에이전트 워크플로를 구성할 때 사용할 수 있는 프레임워크 중 하나입니다.

## 이 장에서 다루는 내용

1. 에이전트 개념 복습
2. 단일 에이전트와 다중 에이전트 차이
3. 다중 에이전트가 필요한 이유
4. 대표적인 에이전트 역할
5. AutoGen 개념
6. 설치와 기본 준비
7. 간단한 역할 분담 예시
8. 로컬 함수로 다중 에이전트 흉내 내기
9. LangChain으로 역할별 체인 구성하기
10. AutoGen 스타일 워크플로 이해하기
11. Gradio로 다중 에이전트 데모 만들기
12. 다중 에이전트 설계 시 주의점

이 장은 `LLM/llm.ipynb`의 LangChain 체인과 Agent 개념, 13장의 LangGraph 상태 기반 흐름을 여러 역할의 협업 구조로 확장하는 내용입니다.


## 20.1 에이전트 개념 복습

LLM 에이전트는 단순히 답변만 생성하는 모델이 아니라, 목표를 달성하기 위해 필요한 행동을 선택하고 실행하는 구조를 의미합니다.

일반적인 LLM 호출은 다음과 같습니다.

```text
질문 -> LLM -> 답변
```

에이전트 구조는 다음처럼 도구 선택이나 중간 판단이 들어갑니다.

```text
목표 입력
  -> LLM이 다음 행동 판단
  -> 검색, 계산, DB 조회, 코드 실행 같은 도구 사용
  -> 결과 관찰
  -> 다시 판단
  -> 최종 답변
```

다중 에이전트는 이 구조를 여러 역할로 나누어 수행합니다.


## 20.2 단일 에이전트와 다중 에이전트 차이

| 구분 | 단일 에이전트 | 다중 에이전트 |
|---|---|---|
| 구성 | 하나의 에이전트가 대부분의 일을 처리 | 여러 에이전트가 역할을 나눔 |
| 장점 | 구조가 단순하고 구현이 쉬움 | 복잡한 문제를 역할별로 나눌 수 있음 |
| 단점 | 한 에이전트에 역할이 과도하게 몰림 | 조율 비용과 실행 시간이 증가 |
| 적합한 작업 | 단순 질의응답, 간단한 도구 사용 | 기획, 작성, 검토, 코드 생성, 토론 |
| 관리 포인트 | 프롬프트와 도구 선택 | 역할, 대화 순서, 종료 조건, 충돌 해결 |

예를 들어 자기소개서 도우미를 다중 에이전트로 만들면 다음처럼 역할을 나눌 수 있습니다.

- 문항 분석 에이전트
- 문장 개선 에이전트
- 면접 질문 생성 에이전트
- 최종 검토 에이전트


## 20.3 다중 에이전트가 필요한 이유

다중 에이전트는 다음 상황에서 유용합니다.

- 한 번의 LLM 호출로 해결하기 어려운 복잡한 문제
- 기획, 작성, 검토처럼 역할이 명확히 나뉘는 작업
- 생성 결과를 다른 관점에서 검토해야 하는 작업
- 코드 작성 후 테스트 계획을 따로 세워야 하는 작업
- RAG 검색 결과를 분석하고 답변 품질을 평가해야 하는 작업
- 데이터 분석에서 분석가, 검토자, 보고서 작성자의 역할을 나누고 싶은 경우

다만 역할을 많이 나눌수록 항상 좋아지는 것은 아닙니다. 호출 비용, 응답 시간, 대화 관리 복잡도가 함께 증가합니다.


## 20.4 대표적인 에이전트 역할

다중 에이전트 시스템에서는 역할 정의가 매우 중요합니다.

| 역할 | 주요 기능 |
|---|---|
| Planner | 문제를 단계로 나누고 실행 계획을 세움 |
| Researcher | 필요한 정보나 문서를 검색하고 정리 |
| Writer | 초안 작성 또는 답변 생성 |
| Reviewer | 결과의 오류, 누락, 위험 요소 검토 |
| Executor | 코드 실행, DB 조회, 도구 호출 수행 |
| Summarizer | 여러 결과를 최종 요약으로 정리 |

역할이 겹치면 대화가 길어지고 결과가 산만해질 수 있습니다. 각 에이전트가 맡을 일과 맡지 않을 일을 분명히 정하는 것이 좋습니다.


## 20.5 AutoGen 개념

AutoGen은 여러 에이전트를 정의하고, 이들이 서로 메시지를 주고받으며 작업을 수행하도록 구성하는 프레임워크입니다.

AutoGen의 핵심 아이디어는 다음과 같습니다.

- 에이전트마다 역할과 시스템 메시지를 지정합니다.
- 에이전트들이 서로 대화하며 작업을 진행합니다.
- 일부 에이전트는 코드 실행이나 도구 호출을 담당할 수 있습니다.
- 대화 종료 조건을 설정해 무한 반복을 방지합니다.
- 사람 사용자가 중간에 개입하거나 승인할 수 있습니다.

이 장에서는 AutoGen의 완전한 운영 환경보다, 다중 에이전트가 어떤 구조로 생각하면 되는지 이해하는 데 초점을 둡니다.


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - 설치 준비
# 이 셀은 다중 에이전트와 AutoGen 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# pyautogen은 AutoGen 기반 다중 에이전트 구성을 실습할 때 사용할 수 있는 패키지입니다.
!pip install pyautogen

# langchain은 역할별 LLM 체인을 구성하는 예시에 사용합니다.
!pip install langchain

# langchain-ollama는 Ollama 로컬 LLM을 LangChain 방식으로 호출할 때 사용합니다.
!pip install langchain-ollama

# gradio는 다중 에이전트 데모 UI를 만들 때 사용합니다.
!pip install gradio


## 20.6 로컬 함수로 다중 에이전트 흉내 내기

프레임워크를 바로 사용하기 전에, Python 함수만으로도 다중 에이전트의 기본 구조를 이해할 수 있습니다.

아래 예시는 하나의 주제를 입력하면 세 역할이 순서대로 작업합니다.

```text
Planner -> Writer -> Reviewer -> Finalizer
```

각 함수는 실제 에이전트처럼 자기 역할에 맞는 결과를 만듭니다.


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - 함수 기반 역할 분담
# 이 셀은 간단한 Python 함수로 다중 에이전트 흐름을 흉내 냅니다.

# planner_agent 함수는 주어진 목표를 단계별 계획으로 나눕니다.
def planner_agent(goal: str) -> str:
    # goal이 비어 있으면 기본 안내 문장을 반환합니다.
    if not goal:
        # 목표가 없다는 메시지입니다.
        return "목표가 입력되지 않았습니다."

    # 목표를 해결하기 위한 계획 문자열을 만듭니다.
    plan = f"목표: {goal}\n1. 핵심 요구사항을 파악한다.\n2. 필요한 내용을 구성한다.\n3. 초안을 작성한다.\n4. 오류와 누락을 검토한다."

    # 계획 문자열을 반환합니다.
    return plan


# writer_agent 함수는 계획을 바탕으로 초안을 작성합니다.
def writer_agent(goal: str, plan: str) -> str:
    # goal과 plan을 바탕으로 초안 문자열을 만듭니다.
    draft = f"[{goal}]에 대한 초안입니다.\n\n{plan}\n\n이 계획을 바탕으로 핵심 내용을 설명하는 답변을 작성합니다."

    # 초안 문자열을 반환합니다.
    return draft


# reviewer_agent 함수는 초안의 보완점을 검토합니다.
def reviewer_agent(draft: str) -> str:
    # 초안에 대한 검토 의견을 작성합니다.
    review = f"검토 결과:\n- 초안의 구조는 적절합니다.\n- 예시를 추가하면 더 이해하기 쉽습니다.\n- 결론을 명확히 정리하면 좋습니다.\n\n검토 대상:\n{draft}"

    # 검토 의견을 반환합니다.
    return review


# finalizer_agent 함수는 초안과 검토 의견을 바탕으로 최종 답변을 만듭니다.
def finalizer_agent(draft: str, review: str) -> str:
    # 최종 답변 문자열을 구성합니다.
    final_answer = f"최종 답변:\n{draft}\n\n반영한 검토 의견:\n{review}"

    # 최종 답변을 반환합니다.
    return final_answer


# run_simple_multi_agent 함수는 네 역할을 순서대로 실행합니다.
def run_simple_multi_agent(goal: str) -> str:
    # Planner 역할을 실행해 계획을 만듭니다.
    plan = planner_agent(goal)

    # Writer 역할을 실행해 초안을 만듭니다.
    draft = writer_agent(goal, plan)

    # Reviewer 역할을 실행해 검토 의견을 만듭니다.
    review = reviewer_agent(draft)

    # Finalizer 역할을 실행해 최종 답변을 만듭니다.
    final_answer = finalizer_agent(draft, review)

    # 최종 답변을 반환합니다.
    return final_answer


# 예시 목표를 입력합니다.
sample_goal = "RAG 챗봇의 장점을 설명하는 발표 자료 만들기"

# 함수 기반 다중 에이전트 흐름을 실행합니다.
simple_result = run_simple_multi_agent(sample_goal)

# 최종 결과를 출력합니다.
print(simple_result)


## 20.7 LangChain으로 역할별 체인 구성하기

실제 LLM을 사용하려면 각 역할을 다른 프롬프트로 구성할 수 있습니다.

예를 들어 다음처럼 만들 수 있습니다.

- Planner Chain: 작업 계획 생성
- Writer Chain: 초안 작성
- Reviewer Chain: 검토와 개선점 제안
- Finalizer Chain: 최종 답변 정리

이 방식은 AutoGen을 쓰기 전에도 다중 에이전트의 역할 분담 효과를 실습할 수 있는 간단한 방법입니다.


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - LangChain 역할별 체인 import
# 이 셀은 역할별 LLM 체인을 구성하기 위한 LangChain 모듈을 불러옵니다.

# ChatPromptTemplate은 역할별 시스템 메시지와 사용자 메시지를 가진 프롬프트를 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 LLM 응답을 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama 로컬 LLM을 LangChain 방식으로 호출합니다.
from langchain_ollama import ChatOllama

# llm 변수에는 사용할 Ollama 모델 이름을 지정합니다.
llm = ChatOllama(model="llama3.1")


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - 역할별 프롬프트와 체인
# 이 셀은 Planner, Writer, Reviewer, Finalizer 역할의 체인을 만듭니다.

# planner_prompt는 목표를 단계별 계획으로 바꾸는 프롬프트입니다.
planner_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 Planner 역할을 지정합니다.
    ("system", "너는 복잡한 작업을 단계별 계획으로 나누는 Planner 에이전트야."),
    # human 메시지는 사용자의 목표를 전달합니다.
    ("human", "다음 목표를 달성하기 위한 실행 계획을 5단계로 작성해줘.\n목표: {goal}"),
])

# writer_prompt는 계획을 바탕으로 초안을 작성하는 프롬프트입니다.
writer_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 Writer 역할을 지정합니다.
    ("system", "너는 계획을 바탕으로 명확한 초안을 작성하는 Writer 에이전트야."),
    # human 메시지는 목표와 계획을 전달합니다.
    ("human", "목표: {goal}\n계획:\n{plan}\n\n이 계획을 바탕으로 초안을 작성해줘."),
])

# reviewer_prompt는 초안을 검토하는 프롬프트입니다.
reviewer_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 Reviewer 역할을 지정합니다.
    ("system", "너는 초안의 오류, 누락, 개선점을 찾는 Reviewer 에이전트야."),
    # human 메시지는 검토할 초안을 전달합니다.
    ("human", "다음 초안을 검토하고 개선점을 구체적으로 제안해줘.\n\n{draft}"),
])

# finalizer_prompt는 초안과 검토 의견을 반영해 최종본을 만드는 프롬프트입니다.
finalizer_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 Finalizer 역할을 지정합니다.
    ("system", "너는 초안과 검토 의견을 반영해 최종 답변을 완성하는 Finalizer 에이전트야."),
    # human 메시지는 초안과 검토 의견을 전달합니다.
    ("human", "초안:\n{draft}\n\n검토 의견:\n{review}\n\n검토 의견을 반영한 최종 답변을 작성해줘."),
])

# planner_chain은 Planner 프롬프트, LLM, 출력 파서를 연결합니다.
planner_chain = planner_prompt | llm | StrOutputParser()

# writer_chain은 Writer 프롬프트, LLM, 출력 파서를 연결합니다.
writer_chain = writer_prompt | llm | StrOutputParser()

# reviewer_chain은 Reviewer 프롬프트, LLM, 출력 파서를 연결합니다.
reviewer_chain = reviewer_prompt | llm | StrOutputParser()

# finalizer_chain은 Finalizer 프롬프트, LLM, 출력 파서를 연결합니다.
finalizer_chain = finalizer_prompt | llm | StrOutputParser()


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - LangChain 다중 역할 실행 함수
# 이 셀은 역할별 체인을 순서대로 실행하는 함수를 정의합니다.

# run_langchain_multi_agent 함수는 목표를 받아 계획, 초안, 검토, 최종 답변을 생성합니다.
def run_langchain_multi_agent(goal: str) -> tuple[str, str, str, str]:
    # 목표가 비어 있으면 안내 메시지를 네 출력에 반환합니다.
    if not goal or not goal.strip():
        # 빈 목표에 대한 안내 메시지입니다.
        message = "목표를 입력해 주세요."

        # 네 개 결과 영역에 같은 메시지를 반환합니다.
        return message, message, message, message

    # Planner 체인을 실행해 계획을 생성합니다.
    plan = planner_chain.invoke({"goal": goal})

    # Writer 체인을 실행해 초안을 생성합니다.
    draft = writer_chain.invoke({"goal": goal, "plan": plan})

    # Reviewer 체인을 실행해 검토 의견을 생성합니다.
    review = reviewer_chain.invoke({"draft": draft})

    # Finalizer 체인을 실행해 최종 답변을 생성합니다.
    final = finalizer_chain.invoke({"draft": draft, "review": review})

    # 계획, 초안, 검토, 최종 답변을 반환합니다.
    return plan, draft, review, final


## 20.8 AutoGen 스타일 워크플로 이해하기

AutoGen에서는 에이전트를 객체로 만들고, 각 에이전트가 메시지를 주고받게 구성합니다.

개념적으로는 다음 흐름입니다.

```text
UserProxyAgent
  -> PlannerAgent
  -> WriterAgent
  -> ReviewerAgent
  -> 최종 응답
```

일부 에이전트는 사람이 입력하는 사용자 역할을 대리하고, 일부 에이전트는 LLM으로 답변을 생성하고, 일부 에이전트는 코드 실행이나 도구 호출을 담당합니다.

AutoGen을 실제로 사용할 때는 모델 설정, API 키, 코드 실행 보안, 종료 조건을 반드시 확인해야 합니다.


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - AutoGen 개념 코드 예시
# 이 셀은 AutoGen 스타일의 구성 개념을 보여주는 예시입니다.
# 실제 실행에는 모델 설정과 API 키 또는 로컬 모델 연동 설정이 필요할 수 있습니다.

# autogen 패키지는 AutoGen 에이전트 구성을 위해 사용합니다.
import autogen

# llm_config는 AutoGen 에이전트가 사용할 모델 설정을 담는 딕셔너리입니다.
llm_config = {
    # config_list에는 사용할 모델과 인증 정보를 넣습니다.
    "config_list": [
        # 실제 사용 시에는 모델 제공자와 API 키 설정을 넣습니다.
        {"model": "gpt-4", "api_key": "YOUR_API_KEY"}
    ]
}

# planner는 작업 계획을 세우는 AssistantAgent 예시입니다.
planner = autogen.AssistantAgent(
    # name은 에이전트 이름입니다.
    name="Planner",
    # system_message는 에이전트의 역할 지시문입니다.
    system_message="너는 작업을 단계별 계획으로 나누는 Planner야.",
    # llm_config는 사용할 LLM 설정입니다.
    llm_config=llm_config,
)

# writer는 초안을 작성하는 AssistantAgent 예시입니다.
writer = autogen.AssistantAgent(
    # name은 에이전트 이름입니다.
    name="Writer",
    # system_message는 Writer의 역할을 지정합니다.
    system_message="너는 계획을 바탕으로 초안을 작성하는 Writer야.",
    # llm_config는 사용할 LLM 설정입니다.
    llm_config=llm_config,
)

# reviewer는 초안을 검토하는 AssistantAgent 예시입니다.
reviewer = autogen.AssistantAgent(
    # name은 에이전트 이름입니다.
    name="Reviewer",
    # system_message는 Reviewer의 역할을 지정합니다.
    system_message="너는 초안의 오류와 누락을 검토하는 Reviewer야.",
    # llm_config는 사용할 LLM 설정입니다.
    llm_config=llm_config,
)

# user_proxy는 사람 사용자의 입력을 대리하는 에이전트입니다.
user_proxy = autogen.UserProxyAgent(
    # name은 사용자 대리 에이전트 이름입니다.
    name="UserProxy",
    # human_input_mode는 사람이 언제 개입할지 정합니다.
    human_input_mode="NEVER",
    # code_execution_config는 코드 실행을 사용할지 설정합니다.
    code_execution_config=False,
)

# 아래 코드는 실제 대화를 시작하는 예시입니다.
# user_proxy.initiate_chat(planner, message="RAG 챗봇 발표 자료를 만들어줘.")


## 20.9 LangGraph와 다중 에이전트

13장에서 배운 LangGraph도 다중 에이전트 구조를 표현하는 데 사용할 수 있습니다.

각 에이전트를 노드로 보고, 실행 순서를 엣지로 연결하면 다음처럼 구성할 수 있습니다.

```text
START
  -> Planner Node
  -> Researcher Node
  -> Writer Node
  -> Reviewer Node
  -> Finalizer Node
  -> END
```

조건 분기를 추가하면 Reviewer가 문제가 있다고 판단했을 때 Writer에게 다시 돌려보내는 반복 구조도 만들 수 있습니다.

```text
Reviewer 결과가 통과
  -> Finalizer

Reviewer 결과가 수정 필요
  -> Writer로 되돌림
```

이런 구조에서는 반복 횟수 제한을 반드시 넣어 무한 루프를 방지해야 합니다.


## 20.10 Gradio 다중 에이전트 데모

Gradio를 사용하면 사용자가 목표를 입력하고 Planner, Writer, Reviewer, Finalizer의 결과를 탭별로 볼 수 있는 UI를 만들 수 있습니다.

이 구조는 실제 AutoGen 그룹 채팅은 아니지만, 다중 역할 워크플로를 학습하기에 좋습니다.


In [ ]:
# 교재 위치: 20장 다중 에이전트와 AutoGen - Gradio 다중 에이전트 UI
# 이 셀은 역할별 결과를 탭으로 보여주는 Gradio UI를 구성합니다.

# gradio를 gr이라는 별칭으로 불러옵니다.
import gradio as gr

# Blocks는 여러 컴포넌트를 자유롭게 배치할 수 있는 Gradio 컨테이너입니다.
with gr.Blocks() as multi_agent_demo:
    # Markdown은 화면 제목을 표시합니다.
    gr.Markdown("# 다중 에이전트 워크플로 데모")

    # goal_input은 사용자의 목표를 입력받는 Textbox입니다.
    goal_input = gr.Textbox(label="목표 입력", lines=3, value="RAG 챗봇의 장점을 설명하는 발표 자료 만들기")

    # run_button은 다중 에이전트 실행 버튼입니다.
    run_button = gr.Button("다중 에이전트 실행")

    # Tabs는 역할별 결과를 나누어 보여줍니다.
    with gr.Tabs():
        # Planner 결과 탭입니다.
        with gr.Tab("Planner"):
            # plan_output은 계획 결과를 표시합니다.
            plan_output = gr.Textbox(label="계획", lines=12)

        # Writer 결과 탭입니다.
        with gr.Tab("Writer"):
            # draft_output은 초안 결과를 표시합니다.
            draft_output = gr.Textbox(label="초안", lines=12)

        # Reviewer 결과 탭입니다.
        with gr.Tab("Reviewer"):
            # review_output은 검토 결과를 표시합니다.
            review_output = gr.Textbox(label="검토 의견", lines=12)

        # Finalizer 결과 탭입니다.
        with gr.Tab("Finalizer"):
            # final_output은 최종 답변을 표시합니다.
            final_output = gr.Textbox(label="최종 답변", lines=12)

    # 버튼 클릭 시 run_langchain_multi_agent 함수를 실행합니다.
    run_button.click(
        # fn에는 실행할 다중 역할 함수를 지정합니다.
        fn=run_langchain_multi_agent,
        # inputs에는 목표 입력 Textbox를 지정합니다.
        inputs=goal_input,
        # outputs에는 역할별 출력 Textbox들을 지정합니다.
        outputs=[plan_output, draft_output, review_output, final_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 다중 에이전트 데모를 확인할 수 있습니다.
# multi_agent_demo.launch()


## 20.11 다중 에이전트 설계 시 주의점

다중 에이전트는 강력하지만 설계가 복잡해질 수 있습니다.

주의할 점은 다음과 같습니다.

- 역할을 너무 많이 나누면 실행 시간이 길어집니다.
- 각 에이전트의 책임이 겹치면 결과가 반복되거나 충돌할 수 있습니다.
- Reviewer가 계속 수정을 요구하면 무한 루프가 생길 수 있습니다.
- 종료 조건과 최대 반복 횟수를 설정해야 합니다.
- 도구 실행 에이전트에는 보안 제한이 필요합니다.
- 코드 실행이나 DB 변경 작업은 사람 승인 단계를 두는 것이 좋습니다.
- 비용과 호출 횟수를 추적해야 합니다.
- LangSmith 같은 도구로 각 에이전트의 입력과 출력을 기록하면 디버깅이 쉬워집니다.

학습 단계에서는 먼저 2~4개의 역할로 단순하게 시작하고, 실제로 필요한 경우에만 역할을 늘리는 것이 좋습니다.


## 20.12 활용 예시

다중 에이전트 구조는 다음 실습에 적용할 수 있습니다.

| 활용 분야 | 에이전트 구성 예시 |
|---|---|
| RAG 챗봇 | Retriever, Answerer, Citation Checker, Reviewer |
| DB 챗봇 | SQL Generator, SQL Validator, DB Executor, Result Explainer |
| 자기소개서 도우미 | Question Analyzer, Feedback Coach, Rewriter, Interviewer |
| 코드 생성 | Planner, Coder, Tester, Reviewer |
| 보고서 작성 | Researcher, Analyst, Writer, Editor |
| 교육 챗봇 | Tutor, Quiz Maker, Feedback Agent, Summarizer |

이전 장에서 만든 실습들도 모두 다중 에이전트 구조로 확장할 수 있습니다.


## 20.13 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| 에이전트 대화가 길어짐 | 종료 조건 부족 | 최대 반복 횟수와 종료 문구 설정 |
| 결과가 반복됨 | 역할 정의가 겹침 | 각 에이전트의 책임을 명확히 분리 |
| 비용이 커짐 | 에이전트 수와 호출 횟수 증가 | 필요한 역할만 유지하고 중간 결과 재사용 |
| AutoGen 인증 오류 | API 키 또는 모델 설정 문제 | llm_config의 모델과 키 확인 |
| 코드 실행 위험 | Executor 권한 과도 | 코드 실행 비활성화 또는 sandbox 사용 |
| 응답 품질 낮음 | Planner 결과가 부실 | 계획 단계 프롬프트 개선 |
| 검토가 약함 | Reviewer 기준 부족 | 검토 기준과 체크리스트를 프롬프트에 추가 |


## 20.14 정리

이 장에서는 다중 에이전트와 AutoGen의 기본 개념을 정리했습니다.

핵심 정리는 다음과 같습니다.

- 에이전트는 목표를 달성하기 위해 판단하고 행동하는 LLM 기반 구조입니다.
- 다중 에이전트는 여러 에이전트가 역할을 나누어 협업하는 방식입니다.
- Planner, Writer, Reviewer, Executor, Summarizer 같은 역할을 정의할 수 있습니다.
- AutoGen은 여러 에이전트가 메시지를 주고받으며 작업하는 구조를 만들 수 있는 프레임워크입니다.
- LangChain 체인만으로도 역할별 다중 에이전트 흐름을 실습할 수 있습니다.
- LangGraph를 사용하면 에이전트 흐름을 상태 기반 그래프로 관리할 수 있습니다.
- 다중 에이전트 설계에서는 역할 분리, 종료 조건, 비용, 보안이 중요합니다.

다음 장에서는 전체 실습 프로젝트를 정리합니다. 지금까지 만든 LLM, RAG, DB 챗봇, LangGraph, Gradio, 멀티모달 실습을 하나의 학습 흐름으로 다시 묶습니다.
